## [Pytorch Image Captioning](https://ithelp.ithome.com.tw/articles/10304033?sc=rss.iron)
https://ithelp.ithome.com.tw/articles/10304033?sc=rss.iron

In [1]:
import numpy as np
import pandas as pd
import os
import torch
from torch.utils.data import DataLoader,Dataset
import nltk

In [2]:
#nltk.download('punkt')

In [3]:
class Vocabulary:
    # 這邊是先建立自己的字典，後面才會繼續增加
    def __init__(self, freq_threshold):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    # 把一段文字透過NLTK的tokenizer切成token，再轉成小寫
    @staticmethod
    def tokenizer_eng(text):
        return [tok.lower() for tok in nltk.tokenize.word_tokenize(text)]

    
    def build_vocabulary(self, sentence_list):
        frequencies = {}
        idx = 4

        # 每個句子
        for sentence in sentence_list:
            # 把句子透過tokenizer 轉換成words
            for word in self.tokenizer_eng(sentence):
                if word not in frequencies:
                    frequencies[word] = 1

                else:
                    frequencies[word] += 1
                # words 要出現夠多次才會被加入到 vocabulary
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    # 這個就是文字轉數值的地方，簡單來說文字先看 stoi 裡面有沒有，如果沒有的話就回傳<UNK>的數值
    def numericalize(self, text):
        tokenized_text = self.tokenizer_eng(text)

        return [
            self.stoi[token] if token in self.stoi else self.stoi["<UNK>"]
            for token in tokenized_text
        ]

In [4]:
# 這邊是建立圖片跟文字之間的關係
class FlickrDataset(Dataset):
    def __init__(self, root_dir, captions_file, transform=None, freq_threshold=5):
        self.root_dir = root_dir
        self.df = pd.read_csv(captions_file)
        self.transform = transform

        # 載入圖片跟敘述
        self.imgs = self.df["image"]
        self.captions = self.df["caption"]

        # 然後這邊就是設定 Vocab 的頻率threshold
        self.vocab = Vocabulary(freq_threshold)
        # 然後這邊就是設定把文字的部分丟進去建立字典
        self.vocab.build_vocabulary(self.captions.tolist())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        caption = self.captions[index]
        img_id = self.imgs[index]
        img = Image.open(os.path.join(self.root_dir, img_id)).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)
        # SOS => start of sentence
        numericalized_caption = [self.vocab.stoi["<SOS>"]]
        # 這邊就是轉換成向量
        numericalized_caption += self.vocab.numericalize(caption)
        # EOS => end of sentence
        numericalized_caption.append(self.vocab.stoi["<EOS>"])

        # 因此這裡就是一個影像跟 一排vector 的輸出
        return img, torch.tensor(numericalized_caption)

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

In [6]:
# import torchvision.models as models 還有 weight
from torchvision.models import inception_v3, Inception_V3_Weights

class EncoderCNN(nn.Module):
    def __init__(self, embed_size, train_CNN=False):
        super(EncoderCNN, self).__init__()
        self.train_CNN = train_CNN
        # 載入 inception v3架構及權重
        self.inception = inception_v3(aux_logits=False,init_weights=Inception_V3_Weights.IMAGENET1K_V1)
        # 把最後一層fc的部分換掉成我們的output 為 embed_size，transfer learning
        self.inception.fc = nn.Linear(self.inception.fc.in_features, embed_size)
        for name, param in self.inception.named_parameters():
            if "fc.weight" in name or "fc.bias" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False
                
        self.relu = nn.ReLU()
        #self.times = []
        self.dropout = nn.Dropout(0.5)

    def forward(self, images):
        features = self.inception(images)
        return self.dropout(self.relu(features))

In [7]:
class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers):
        super(DecoderRNN, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(0.5)

    def forward(self, features, captions):
        embeddings = self.dropout(self.embed(captions))
        # 加入 encoder output 的feature 進來
        embeddings = torch.cat((features.unsqueeze(0), embeddings), dim=0)
        # 然後丟出
        hiddens, _ = self.lstm(embeddings)
        outputs = self.linear(hiddens)
        return outputs

In [8]:
class EncoderDecoder(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers):
        super(EncoderDecoder, self).__init__()
        self.encoderCNN = EncoderCNN(embed_size)
        self.decoderRNN = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers)

    def forward(self, images, captions):
        features = self.encoderCNN(images)
        outputs = self.decoderRNN(features, captions)
        return outputs

    def caption_image(self, image, vocabulary, max_length=50):
        result_caption = []

        with torch.no_grad():
          
            x = self.encoderCNN(image).unsqueeze(0)
            states = None

            for _ in range(max_length):
	            # RNN每次的 output 都會把 state 覆蓋
                hiddens, states = self.decoderRNN.lstm(x, states)
                # 然後再丟到 linear 跑及幾率
                output = self.decoderRNN.linear(hiddens.squeeze(0))
                # 找最大的
                predicted = output.argmax(1)
	            # 把結果放置 result_caption
                result_caption.append(predicted.item())
                x = self.decoderRNN.embed(predicted).unsqueeze(0)
				#一直重複上面的過程一直跑到結束token (<EOS>) 出現在停止
                if vocabulary.itos[predicted.item()] == "<EOS>":
                    break

        return [vocabulary.itos[idx] for idx in result_caption]

In [9]:
base_dir = 'C:/Users/leonjye/Documents/DeepLearning/Flickr8k'

In [10]:
from torch.nn.utils.rnn import pad_sequence 
class MyCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        imgs = [item[0].unsqueeze(0) for item in batch]
        imgs = torch.cat(imgs, dim=0)
        targets = [item[1] for item in batch]
        # 把他填充到等長
        targets = pad_sequence(targets, batch_first=False, padding_value=self.pad_idx)

        return imgs, targets

In [11]:
import torchvision.transforms as transforms
# 因為 Inpection V3 的 model input 的影像是299 x 299
transform = transforms.Compose(
    [
        transforms.Resize((356, 356)),
        transforms.RandomCrop((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
#注意inception輸入的圖形大小，應該是224x224吧
dataset = FlickrDataset(
    root_dir = base_dir + "/Images", 
    captions_file = base_dir + '/captions.txt',
    transform=transform)

dataloader = DataLoader(dataset=dataset, 
                        batch_size=32, 
                        shuffle=True,
                        num_workers=8, #印象中在windows會有問題
                        pin_memory=True,
                        collate_fn=MyCollate(pad_idx=dataset.vocab.stoi["<PAD>"]) #將caption裡面的token轉成index，並且pad到相同長度，以便輸�
                        )

In [12]:
#Hyperparams
embed_size = 256
hidden_size = 256
vocab_size = len(dataset.vocab)
num_layers = 1
learning_rate = 3e-4
num_epochs = 2
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
# initialize model, loss etc
model = EncoderDecoder(embed_size, hidden_size, vocab_size, num_layers).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=dataset.vocab.stoi["<PAD>"])
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [14]:
from tqdm import tqdm
model.train()

for epoch in range(num_epochs):
    for idx, (imgs, captions) in (pbar := tqdm(enumerate(dataloader), 
                                               total=len(dataloader))):
        imgs = imgs.to(device)
        captions = captions.to(device)

        outputs = model(imgs, captions[:-1])
        loss = criterion(
            outputs.reshape(-1, outputs.shape[2]), captions.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward(loss)
        optimizer.step()
        pbar.set_postfix(f"Epoch {epoch}")
        pbar.set_description(f'loss= {loss.item()}')